In [1]:
import pandas as pd
import numpy as np

import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from tabulate import tabulate
from SQUIC_functions import *

In [2]:
# chunk_acc = pd.read_csv(f'accounts.csv', chunksize=1000)
# chunk_trns = pd.read_csv(f'transactions.csv', chunksize=1000)

# accounts = pd.concat(chunk_acc)
# transactions = pd.concat(chunk_trns)

accounts = pd.read_csv('accounts.csv')
transactions = pd.read_csv('transactions.csv')

In [3]:
balances = {
    acc_id: [{"date": 0, "balance": round(float(init_bal), 2)}]
    for acc_id, init_bal in zip(accounts["ACCOUNT_ID"], accounts["INIT_BALANCE"])
}

In [4]:
import pandas as pd
from scipy.sparse import lil_matrix

transactions.sort_values(by="TX_ID", inplace=True)

max_account_id = max(transactions["SENDER_ACCOUNT_ID"].max(),
                     transactions["RECEIVER_ACCOUNT_ID"].max()) + 1

matrix = lil_matrix((max_account_id, max_account_id), dtype=int)


for row in transactions.itertuples(index=False):
    orig_acct = row.SENDER_ACCOUNT_ID
    bene_acct = row.RECEIVER_ACCOUNT_ID
    amount = float(row.TX_AMOUNT)  # make sure this is float for balance operations
    tx_type = row.TX_TYPE
    date = row.TIMESTAMP

    # Update sparse matrix
    matrix[orig_acct, bene_acct] += int(amount)

    # Process sender
    if tx_type in ['TRANSFER', 'WITHDRAWAL'] and orig_acct in balances:
        last_balance = balances[orig_acct][-1]["balance"]
        new_balance = last_balance - amount
        balances[orig_acct].append({
            "date": date,
            "balance": round(new_balance, 2)
        })

    # Process receiver
    if tx_type in ['TRANSFER', 'DEPOSIT'] and bene_acct in balances:
        last_balance = balances[bene_acct][-1]["balance"]
        new_balance = last_balance + amount
        balances[bene_acct].append({
            "date": date,
            "balance": round(new_balance, 2)
        })


In [5]:
for i, (acct_id, account_data) in enumerate(balances.items()):
    if i < 1:
        print(acct_id)
        print(account_data)
        break

0
[{'date': 0, 'balance': 184.44}, {'date': 2, 'balance': 253.63}, {'date': 3, 'balance': 322.82}, {'date': 7, 'balance': 374.37}, {'date': 13, 'balance': 443.56}, {'date': 16, 'balance': 351.34}, {'date': 16, 'balance': 259.12}, {'date': 18, 'balance': 310.67}, {'date': 23, 'balance': 379.86}, {'date': 26, 'balance': 287.64}, {'date': 26, 'balance': 195.42}, {'date': 27, 'balance': 103.2}, {'date': 27, 'balance': 10.98}, {'date': 27, 'balance': 62.53}, {'date': 29, 'balance': -29.69}, {'date': 29, 'balance': -121.91}, {'date': 30, 'balance': -52.72}, {'date': 31, 'balance': 16.47}, {'date': 37, 'balance': -75.75}, {'date': 37, 'balance': -167.97}, {'date': 38, 'balance': -116.42}, {'date': 49, 'balance': -47.23}, {'date': 49, 'balance': -139.45}, {'date': 49, 'balance': -231.67}, {'date': 51, 'balance': -162.48}, {'date': 53, 'balance': -93.29}, {'date': 57, 'balance': -185.51}, {'date': 57, 'balance': -277.73}, {'date': 58, 'balance': -226.18}, {'date': 68, 'balance': -318.4}, {'date

In [6]:
import pandas as pd

# Step 1: Flatten the balances dict to a DataFrame
records = []

for user_id, history in balances.items():
    for entry in history:
        records.append({
            "ACCOUNT_ID": user_id,
            "TIMESTAMP": entry["date"],
            "BALANCE": entry["balance"]
        })

df_balances = pd.DataFrame(records)

# Step 2: Sort by timestamp and user, then remove duplicates keeping the latest
df_balances.sort_values(by=["ACCOUNT_ID", "TIMESTAMP"], inplace=True)
df_balances.drop_duplicates(subset=["ACCOUNT_ID", "TIMESTAMP"], keep="last", inplace=True)

# Step 3: Create a full grid of all timestamps and all users
all_dates = sorted(transactions["TIMESTAMP"].unique())
all_users = df_balances["ACCOUNT_ID"].unique()
grid = pd.MultiIndex.from_product([all_users, all_dates], names=["ACCOUNT_ID", "TIMESTAMP"])

# Step 4: Reindex the balance DataFrame to the full grid
df_balances = df_balances.set_index(["ACCOUNT_ID", "TIMESTAMP"])
df_balances = df_balances.reindex(grid)

# Step 5: Forward fill missing balances (per user)
df_balances["BALANCE"] = df_balances["BALANCE"].groupby(level=0).ffill()

# Step 6: Reset index and pivot to get final table: one row per date, one column per user
df_result = df_balances.reset_index().pivot(index="TIMESTAMP", columns="ACCOUNT_ID", values="BALANCE")
df_result.index.name = "date"
df_result.columns = df_result.columns.astype(str)  # match your original str(user) keys

In [7]:
df_result

ACCOUNT_ID,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
date,,,,,,,,,,,,,,,,,,,,,
0,184.44,175.80,142.06,125.89,151.13,140.49,178.38,130.33,147.66,158.34,...,411.76,331.00,340.34,426.02,463.36,411.81,437.84,238.28,386.67,297.29
1,184.44,0.00,142.06,125.89,182.79,140.49,194.34,130.33,163.62,31.70,...,411.76,346.06,356.30,426.02,463.36,411.81,452.90,238.28,386.67,297.29
2,253.63,12.38,142.06,138.27,195.17,217.43,194.34,130.33,163.62,92.89,...,485.33,425.57,356.30,426.02,475.74,578.08,452.90,307.47,416.62,12.55
3,322.82,12.38,142.06,138.27,195.17,217.43,264.42,164.84,163.62,92.89,...,517.69,425.57,356.30,461.13,475.74,647.27,452.90,406.85,551.40,147.33
4,322.82,22.28,198.79,138.27,283.92,76.94,264.42,202.88,163.62,92.89,...,758.87,493.80,137.97,478.19,592.32,702.29,165.99,637.58,681.07,272.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,-1312.03,-3998.92,-1015.09,-929.73,759.18,175.20,-68.40,131.37,-953.47,-445.13,...,6129.84,2741.40,6605.25,3671.52,1318.73,5914.94,7758.35,9444.49,11741.15,11217.91
196,-1312.03,-3998.92,-1157.14,-929.73,759.18,175.20,-68.40,131.37,-953.47,-445.13,...,6129.84,2754.61,6605.25,3709.68,1318.73,5932.86,7897.52,9457.70,11788.50,11285.84
197,-1312.03,-3998.92,-1299.19,-929.73,639.73,175.20,-68.40,131.37,-953.47,-571.77,...,5823.71,2800.86,6641.93,3833.55,1418.36,5990.27,8095.31,9607.42,11959.72,11468.65
